# Duplicate Charged Particle Track Detection
### Improved version: Euclidean vs Mahalanobis vs Cosine comparison
**Changes from previous version:**
- All 500 events used for pair generation (was capped at 20)
- Separate dedicated networks for Euclidean, Cosine, and Mahalanobis
- Wider/deeper architecture (128→64→emb_dim) + BatchNorm
- Embedding dimension increased to 16
- Removed final ReLU in Mahalanobis network embeddings
- Contrastive margin increased to 2.0
- Mahalanobis training loop never overwritten by Euclidean re-training
- Gradient clipping on all three training loops
- ReduceLROnPlateau scheduler on all three loops
- V updated from Mahalanobis-tuned embeddings only


In [2]:
import os
import uproot
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
import awkward as ak
import time

# Set seeds for reproducibility
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)
os.environ['PYTHONHASHSEED'] = str(seed_value)

def load_root_file(file_path, branches=None, print_branches=False):
    all_branches = {}
    with uproot.open(file_path) as file:
        tree = file["tree"]
        if branches is None:
            branches = tree.keys()
        if print_branches:
            print("Branches:", tree.keys())
        for branch in branches:
            try:
                all_branches[branch] = (tree[branch].array(library="np"))
            except uproot.KeyInFileError as e:
                print(f"KeyInFileError: {e}")
        all_branches['event'] = tree.num_entries
    return all_branches

branches_list = [
    't5_innerRadius',
    't5_bridgeRadius',
    't5_outerRadius',
    't5_pt',
    't5_eta',
    't5_phi',
    't5_isFake',
    't5_t3_idx0',
    't5_t3_idx1',
    't5_t3_fakeScore1',
    't5_t3_promptScore1',
    't5_t3_displacedScore1',
    't5_t3_fakeScore2',
    't5_t3_promptScore2',
    't5_t3_displacedScore2',
    't5_pMatched',
    't5_sim_vxy',
    't5_sim_vz',
    't5_matched_simIdx'
]

branches_list += [
    'pLS_eta',
    'pLS_etaErr',
    'pLS_phi',
    'pLS_matched_simIdx',
    'pLS_circleCenterX',
    'pLS_circleCenterY',
    'pLS_circleRadius',
    'pLS_ptIn',
    'pLS_ptErr',
    'pLS_px',
    'pLS_py',
    'pLS_pz',
    'pLS_isQuad',
    'pLS_isFake'
]

suffixes = ['r', 'z', 'eta', 'phi', 'layer']
branches_list += [f't5_t3_{i}_{suffix}' for i in [0, 2, 4] for suffix in suffixes]

!pip install gdown

import gdown

file_id = '1l0NvKngLrY7HZxx69ZrnLYEWWvTwmM5f'
url = f'https://drive.google.com/uc?id={file_id}'
file_path = 'data_file.root'

if not os.path.exists(file_path):
    print(f"Downloading {file_path} from Google Drive...")
    gdown.download(url, file_path, quiet=False)
else:
    print(f"File '{file_path}' already exists locally. Skipping download.")

branches = load_root_file(file_path, branches_list, print_branches=True)

File 'data_file.root' already exists locally. Skipping download.
Branches: ['sim_pt', 'sim_eta', 'sim_phi', 'sim_pca_dxy', 'sim_pca_dz', 'sim_q', 'sim_event', 'sim_pdgId', 'sim_vx', 'sim_vy', 'sim_vz', 'sim_trkNtupIdx', 'sim_TC_matched', 'sim_TC_matched_mask', 'tc_pt', 'tc_eta', 'tc_phi', 'tc_type', 'tc_isFake', 'tc_isDuplicate', 'tc_matched_simIdx', 'sim_dummy', 'tc_dummy', 'pT5_matched_simIdx', 'pT5_hitIdxs', 'sim_pT5_matched', 'pT5_pt', 'pT5_eta', 'pT5_phi', 'pT5_isFake', 't5_sim_vxy', 't5_sim_vz', 'pT5_isDuplicate', 'pT5_score', 'pT5_layer_binary', 'pT5_moduleType_binary', 'pT5_matched_pt', 'pT5_rzChiSquared', 'pT5_rPhiChiSquared', 'pT5_rPhiChiSquaredInwards', 'sim_pT3_matched', 'pT3_pt', 'pT3_isFake', 'pT3_isDuplicate', 'pT3_eta', 'pT3_phi', 'pT3_score', 'pT3_foundDuplicate', 'pT3_matched_simIdx', 'pT3_hitIdxs', 'pT3_pixelRadius', 'pT3_pixelRadiusError', 'pT3_matched_pt', 'pT3_tripletRadius', 'pT3_rPhiChiSquared', 'pT3_rPhiChiSquaredInwards', 'pT3_rzChiSquared', 'pT3_layer_binary'

In [3]:
z_max = np.max([np.max(event) for event in branches[f't5_t3_4_z']])
r_max = np.max([np.max(event) for event in branches[f't5_t3_4_r']])
eta_max = 2.5
phi_max = np.pi
n_events = np.shape(branches['t5_pt'])[0]

# Cap at 500 events
N_EVENTS_USE = min(500, n_events)
print(f'Total events in file: {n_events}, using: {N_EVENTS_USE}')
print(f'Z max: {z_max}, R max: {r_max}, Eta max: {eta_max}')

def delta_phi(phi1, phi2):
    delta = phi1 - phi2
    if delta > np.pi:
        delta -= 2 * np.pi
    elif delta < -np.pi:
        delta += 2 * np.pi
    return delta

Total events in file: 500, using: 500
Z max: 267.2349853515625, R max: 110.10993957519531, Eta max: 2.5


In [4]:
pMATCHED_THRESHOLD = 0.
print(f"\nBuilding T5 features (pMatched >= {pMATCHED_THRESHOLD}) ...")

features_per_event    = []
eta_per_event         = []
displaced_per_event   = []
sim_indices_per_event = []

kept_tot, init_tot = 0, 0
for ev in range(N_EVENTS_USE):

    n_t5 = len(branches['t5_t3_idx0'][ev])
    init_tot += n_t5
    if n_t5 == 0:
        continue

    feat_evt = []
    eta_evt  = []
    sim_evt  = []
    disp_evt = []

    for i in range(n_t5):
        if branches['t5_pMatched'][ev][i] < pMATCHED_THRESHOLD:
            continue

        idx0 = branches['t5_t3_idx0'][ev][i]
        idx1 = branches['t5_t3_idx1'][ev][i]

        eta1 = (branches['t5_t3_0_eta'][ev][idx0])
        eta2 = abs(branches['t5_t3_2_eta'][ev][idx0])
        eta3 = abs(branches['t5_t3_4_eta'][ev][idx0])
        eta4 = abs(branches['t5_t3_2_eta'][ev][idx1])
        eta5 = abs(branches['t5_t3_4_eta'][ev][idx1])

        phi1 = branches['t5_t3_0_phi'][ev][idx0]
        phi2 = branches['t5_t3_2_phi'][ev][idx0]
        phi3 = branches['t5_t3_4_phi'][ev][idx0]
        phi4 = branches['t5_t3_2_phi'][ev][idx1]
        phi5 = branches['t5_t3_4_phi'][ev][idx1]

        z1 = abs(branches['t5_t3_0_z'][ev][idx0])
        z2 = abs(branches['t5_t3_2_z'][ev][idx0])
        z3 = abs(branches['t5_t3_4_z'][ev][idx0])
        z4 = abs(branches['t5_t3_2_z'][ev][idx1])
        z5 = abs(branches['t5_t3_4_z'][ev][idx1])

        r1 = branches['t5_t3_0_r'][ev][idx0]
        r2 = branches['t5_t3_2_r'][ev][idx0]
        r3 = branches['t5_t3_4_r'][ev][idx0]
        r4 = branches['t5_t3_2_r'][ev][idx1]
        r5 = branches['t5_t3_4_r'][ev][idx1]

        inR  = branches['t5_innerRadius' ][ev][i]
        brR  = branches['t5_bridgeRadius'][ev][i]
        outR = branches['t5_outerRadius' ][ev][i]

        s1_fake   = branches['t5_t3_fakeScore1'     ][ev][i]
        s1_prompt = branches['t5_t3_promptScore1'   ][ev][i]
        s1_disp   = branches['t5_t3_displacedScore1'][ev][i]
        d_fake    = branches['t5_t3_fakeScore2'     ][ev][i] - s1_fake
        d_prompt  = branches['t5_t3_promptScore2'   ][ev][i] - s1_prompt
        d_disp    = branches['t5_t3_displacedScore2'][ev][i] - s1_disp

        f = [
            eta1 / eta_max,
            np.cos(phi1),
            np.sin(phi1),
            z1 / z_max,
            r1 / r_max,

            eta2 - abs(eta1),
            delta_phi(phi2, phi1),
            (z2 - z1) / z_max,
            (r2 - r1) / r_max,

            eta3 - eta2,
            delta_phi(phi3, phi2),
            (z3 - z2) / z_max,
            (r3 - r2) / r_max,

            eta4 - eta3,
            delta_phi(phi4, phi3),
            (z4 - z3) / z_max,
            (r4 - r3) / r_max,

            eta5 - eta4,
            delta_phi(phi5, phi4),
            (z5 - z4) / z_max,
            (r5 - r4) / r_max,

            1.0 / inR,
            1.0 / brR,
            1.0 / outR,

            s1_fake, s1_prompt, s1_disp,
            d_fake,  d_prompt,  d_disp
        ]
        feat_evt.append(f)
        eta_evt.append(eta1)
        disp_evt.append(branches['t5_sim_vxy'][ev][i])

        simIdx_list = branches['t5_matched_simIdx'][ev][i]
        sim_evt.append(simIdx_list[0] if len(simIdx_list) else -1)

    if feat_evt:
        features_per_event.append(np.asarray(feat_evt, dtype=np.float32))
        eta_per_event.append(np.asarray(eta_evt,  dtype=np.float32))
        displaced_per_event.append(np.asarray(disp_evt, dtype=np.float32))
        sim_indices_per_event.append(np.asarray(sim_evt, dtype=np.int64))
        kept_tot += len(feat_evt)

print(f"\nKept {kept_tot} / {init_tot} T5s "
      f"({kept_tot/init_tot*100:.2f} %) that passed the pMatched cut.")
print(f"Total events with >=1 kept T5: {len(features_per_event)}")


Building T5 features (pMatched >= 0.0) ...

Kept 3630781 / 3630781 T5s (100.00 %) that passed the pMatched cut.
Total events with >=1 kept T5: 500


In [5]:
KEEP_FRAC_PLS = 0.40
print(f"\nBuilding pLS features ...")

pLS_features_per_event    = []
pLS_eta_per_event         = []
pLS_sim_indices_per_event = []

kept_tot_pls, init_tot_pls = 0, 0
for ev in range(N_EVENTS_USE):
    n_pls = len(branches['pLS_eta'][ev])
    init_tot_pls += n_pls
    if n_pls == 0:
        continue

    feat_evt, eta_evt, sim_evt = [], [], []

    for i in range(n_pls):
        if branches['pLS_isFake'][ev][i]:
            continue
        if np.random.random() > KEEP_FRAC_PLS:
            continue

        eta = branches['pLS_eta'][ev][i]
        etaErr = branches['pLS_etaErr'][ev][i]
        phi = branches['pLS_phi'][ev][i]
        circleCenterX = np.abs(branches['pLS_circleCenterX'][ev][i])
        circleCenterY = np.abs(branches['pLS_circleCenterY'][ev][i])
        circleRadius = branches['pLS_circleRadius'][ev][i]
        ptIn = branches['pLS_ptIn'][ev][i]
        ptErr = branches['pLS_ptErr'][ev][i]
        isQuad = branches['pLS_isQuad'][ev][i]

        f = [
            eta/4.0,
            etaErr/.00139,
            np.cos(phi),
            np.sin(phi),
            1.0 / ptIn,
            np.log10(ptErr),
            isQuad,
            np.log10(circleCenterX),
            np.log10(circleCenterY),
            np.log10(circleRadius),
        ]

        feat_evt.append(f)
        eta_evt.append(eta)

        sim_list = branches['pLS_matched_simIdx'][ev][i]
        sim_evt.append(sim_list[0] if len(sim_list) else -1)

    if feat_evt:
        pLS_features_per_event   .append(np.asarray(feat_evt, dtype=np.float32))
        pLS_eta_per_event        .append(np.asarray(eta_evt,  dtype=np.float32))
        pLS_sim_indices_per_event.append(np.asarray(sim_evt, dtype=np.int64))
        kept_tot_pls += len(feat_evt)

print(f"\nKept {kept_tot_pls} / {init_tot_pls} pLSs "
      f"({kept_tot_pls/init_tot_pls*100:.2f} %) that passed the selections.")
print(f"Total events with >=1 kept pLS: {len(pLS_features_per_event)}")


Building pLS features ...

Kept 3722110 / 11314631 pLSs (32.90 %) that passed the selections.
Total events with >=1 kept pLS: 500


In [6]:
import time, random, math, numpy as np
from concurrent.futures import ProcessPoolExecutor
from sklearn.model_selection import train_test_split

DELTA_R2_CUT = 0.02

def _pairs_single_event(evt_idx, F, S, D, max_sim, max_dis, invalid_sim):
    n = F.shape[0]
    if n < 2:
        return evt_idx, [], []
    eta1 = F[:, 0] * eta_max
    phi1 = np.arctan2(F[:, 2], F[:, 1])
    idx_l, idx_r = np.triu_indices(n, k=1)
    idxs_triu = np.stack((idx_l, idx_r), axis=-1)
    simidx_l = S[idx_l]
    simidx_r = S[idx_r]
    eta_l = eta1[idx_l]
    eta_r = eta1[idx_r]
    phi_l = phi1[idx_l]
    phi_r = phi1[idx_r]
    dphi = np.abs(phi_l - phi_r)
    dphi[dphi > np.pi] -= 2 * np.pi
    dr2 = (eta_l - eta_r)**2 + dphi**2
    dr2_valid = (dr2 < DELTA_R2_CUT)
    sim_idx_same = (simidx_l == simidx_r)
    sim_mask = dr2_valid & sim_idx_same & (simidx_l != invalid_sim)
    dis_mask = dr2_valid & ~sim_idx_same
    sim_pairs = idxs_triu[sim_mask]
    dis_pairs = idxs_triu[dis_mask]
    random.seed(evt_idx)
    if len(sim_pairs) > max_sim:
        sim_pairs = sim_pairs[random.sample(range(len(sim_pairs)), max_sim)]
    if len(dis_pairs) > max_dis:
        dis_pairs = dis_pairs[random.sample(range(len(dis_pairs)), max_dis)]
    print(f"[evt {evt_idx:4d}]  T5s={n:5d}  sim={len(sim_pairs):3d}  dis={len(dis_pairs):3d}")
    return evt_idx, sim_pairs, dis_pairs

def create_t5_pairs_balanced_parallel(
        features_per_event, sim_indices_per_event, displaced_per_event,
        *, max_similar_pairs_per_event=100, max_dissimilar_pairs_per_event=450,
        invalid_sim_idx=-1, n_workers=1, n_events_use=None):
    t0 = time.time()
    n_use = n_events_use if n_events_use is not None else len(features_per_event)
    print(f"\n>>> Pair generation (DR2 < {DELTA_R2_CUT}) over {n_use} events")
    work_args = [
        (evt_idx, features_per_event[evt_idx], sim_indices_per_event[evt_idx],
         displaced_per_event[evt_idx], max_similar_pairs_per_event,
         max_dissimilar_pairs_per_event, invalid_sim_idx)
        for evt_idx in range(n_use)
    ]
    sim_L, sim_R, sim_disp = [], [], []
    dis_L, dis_R, dis_disp = [], [], []
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = [pool.submit(_pairs_single_event, *args) for args in work_args]
        for fut in futures:
            evt_idx, sim_pairs_evt, dis_pairs_evt = fut.result()
            F = features_per_event[evt_idx]
            D = displaced_per_event[evt_idx]
            for i, j in sim_pairs_evt:
                sim_L.append(F[i]); sim_R.append(F[j])
                sim_disp.append(D[i] > 0.1 or D[j] > 0.1)
            for i, j in dis_pairs_evt:
                dis_L.append(F[i]); dis_R.append(F[j])
                dis_disp.append(D[i] > 0.1 or D[j] > 0.1)
    X_left  = np.concatenate([np.asarray(sim_L, dtype=np.float32), np.asarray(dis_L, dtype=np.float32)], axis=0)
    X_right = np.concatenate([np.asarray(sim_R, dtype=np.float32), np.asarray(dis_R, dtype=np.float32)], axis=0)
    y       = np.concatenate([np.zeros(len(sim_L), dtype=np.int32), np.ones(len(dis_L), dtype=np.int32)])
    disp_flag = np.concatenate([np.asarray(sim_disp, dtype=bool), np.asarray(dis_disp, dtype=bool)], axis=0)
    print(f"<<< done in {time.time() - t0:.1f}s  | sim {len(sim_L)}  dis {len(dis_L)}  total {len(y)}")
    return X_left, X_right, y, disp_flag

X_left, X_right, y_t5, disp_t5 = create_t5_pairs_balanced_parallel(
    features_per_event, sim_indices_per_event, displaced_per_event,
    max_similar_pairs_per_event=1000, max_dissimilar_pairs_per_event=1000,
    invalid_sim_idx=-1, n_workers=1, n_events_use=len(features_per_event)
)

if len(y_t5) == 0:
    raise ValueError("No T5 pairs generated. Check filters/data.")

mask = (np.isfinite(X_left).all(axis=1) & np.isfinite(X_right).all(axis=1))
if not mask.all():
    print(f"Filtering {np.sum(~mask)} pairs with NaN/Inf")
    X_left, X_right, y_t5, disp_t5 = X_left[mask], X_right[mask], y_t5[mask], disp_t5[mask]

weights_t5 = np.where(disp_t5, 5.0, 1.0).astype(np.float32)

X_left_train, X_left_test, X_right_train, X_right_test, y_t5_train, y_t5_test, w_t5_train, w_t5_test = train_test_split(
    X_left, X_right, y_t5, weights_t5, test_size=0.20, random_state=42, stratify=y_t5, shuffle=True
)

pct_disp = np.mean(disp_t5) * 100
print(f"{pct_disp:.2f}% of all T5 pairs involve a displaced T5")


>>> Pair generation (DR2 < 0.02) over 500 events
[evt    0]  T5s= 7083  sim=1000  dis=1000
[evt    1]  T5s= 6400  sim=1000  dis=1000
[evt    2]  T5s= 6708  sim=1000  dis=1000
[evt    3]  T5s= 7768  sim=1000  dis=1000
[evt    4]  T5s= 5870  sim=1000  dis=1000
[evt    5]  T5s= 7422  sim=1000  dis=1000
[evt    6]  T5s= 6374  sim=1000  dis=1000
[evt    7]  T5s= 9034  sim=1000  dis=1000
[evt    8]  T5s= 8741  sim=1000  dis=1000
[evt    9]  T5s= 7114  sim=1000  dis=1000
[evt   10]  T5s= 6965  sim=1000  dis=1000
[evt   11]  T5s= 6277  sim=1000  dis=1000
[evt   12]  T5s= 7587  sim=1000  dis=1000
[evt   13]  T5s= 8697  sim=1000  dis=1000
[evt   14]  T5s= 6359  sim=1000  dis=1000
[evt   15]  T5s= 8403  sim=1000  dis=1000
[evt   16]  T5s= 5981  sim=1000  dis=1000
[evt   17]  T5s= 5930  sim=1000  dis=1000
[evt   18]  T5s= 7760  sim=1000  dis=1000
[evt   19]  T5s= 7765  sim=1000  dis=1000
[evt   20]  T5s= 6925  sim=1000  dis=1000
[evt   21]  T5s= 7230  sim=1000  dis=1000
[evt   22]  T5s= 4984  sim

In [7]:
DELTA_R2_CUT_PLS_T5 = 0.02
DISP_VXY_CUT        = 0.1
INVALID_SIM_IDX     = -1
MAX_SIM             = 1000
MAX_DIS             = 1000

def _pairs_pLS_T5_single(evt_idx, F_pLS, S_pLS, F_T5, S_T5, D_T5, max_sim, max_dis, invalid_sim):
    n_p, n_t = F_pLS.shape[0], F_T5.shape[0]
    if n_p == 0 or n_t == 0:
        print(f"[evt {evt_idx:4d}]  pLSs={n_p:5d}  T5s={n_t:5d}  sim={0:4d}  dis={0:4d}")
        return evt_idx, []
    eta_p = F_pLS[:,0] * 4.0
    phi_p = np.arctan2(F_pLS[:,3], F_pLS[:,2])
    eta_t = F_T5[:,0] * eta_max
    phi_t = np.arctan2(F_T5[:,2], F_T5[:,1])
    idx_p, idx_t = np.indices((n_p, n_t))
    idx_p, idx_t = idx_p.flatten(), idx_t.flatten()
    dphi = (phi_p[idx_p] - phi_t[idx_t] + np.pi) % (2 * np.pi) - np.pi
    dr2 = (eta_p[idx_p] - eta_t[idx_t])**2 + dphi**2
    dr2_valid = (dr2 < DELTA_R2_CUT_PLS_T5)
    simidx_p = S_pLS[idx_p]
    simidx_t = S_T5[idx_t]
    sim_idx_same = (simidx_p == simidx_t)
    sim_mask = dr2_valid & sim_idx_same & (simidx_p != invalid_sim)
    dis_mask = dr2_valid & ~sim_idx_same
    sim_pairs = np.column_stack((idx_p[sim_mask], idx_t[sim_mask])) if sim_mask.any() else np.empty((0,2), dtype=int)
    dis_pairs = np.column_stack((idx_p[dis_mask], idx_t[dis_mask])) if dis_mask.any() else np.empty((0,2), dtype=int)
    random.seed(evt_idx)
    if len(sim_pairs) > max_sim:
        sim_pairs = sim_pairs[random.sample(range(len(sim_pairs)), max_sim)]
    if len(dis_pairs) > max_dis:
        dis_pairs = dis_pairs[random.sample(range(len(dis_pairs)), max_dis)]
    print(f"[evt {evt_idx:4d}]  pLSs={n_p:5d}  T5s={n_t:5d}  sim={len(sim_pairs):4d}  dis={len(dis_pairs):4d}")
    packed = []
    for i, j in sim_pairs:
        packed.append((F_pLS[i], F_T5[j], 0, D_T5[j] > DISP_VXY_CUT))
    for i, j in dis_pairs:
        packed.append((F_pLS[i], F_T5[j], 1, D_T5[j] > DISP_VXY_CUT))
    return evt_idx, packed

print(f"\n>>> Building pLS-T5 pairs (DR2 < {DELTA_R2_CUT_PLS_T5}) over {len(features_per_event)} events ...")
t0 = time.time()
all_packed = []
sim_total = 0
dis_total = 0

n_use_pls = min(len(features_per_event), len(pLS_features_per_event))

with ProcessPoolExecutor(max_workers=1) as pool:
    futures = [
        pool.submit(_pairs_pLS_T5_single, ev,
            pLS_features_per_event[ev], pLS_sim_indices_per_event[ev],
            features_per_event[ev], sim_indices_per_event[ev], displaced_per_event[ev],
            MAX_SIM, MAX_DIS, INVALID_SIM_IDX)
        for ev in range(n_use_pls)
    ]
    for fut in futures:
        _, packed = fut.result()
        sim_evt = sum(1 for _, _, lbl, _ in packed if lbl == 0)
        dis_evt = sum(1 for _, _, lbl, _ in packed if lbl == 1)
        sim_total += sim_evt
        dis_total += dis_evt
        all_packed.extend(packed)

print(f"<<< done in {time.time() - t0:.1f}s  | sim {sim_total:5d}  dis {dis_total:5d}  total {len(all_packed):,d}")

pls_feats = np.array([p[0] for p in all_packed], dtype=np.float32)
t5_feats  = np.array([p[1] for p in all_packed], dtype=np.float32)
y_pls     = np.array([p[2] for p in all_packed], dtype=np.int32)
disp_flag = np.array([p[3] for p in all_packed], dtype=bool)
w_pls     = np.array([5.0 if p[3] else 1.0 for p in all_packed], dtype=np.float32)

mask_pls = (np.isfinite(pls_feats).all(axis=1) & np.isfinite(t5_feats).all(axis=1))
if not mask_pls.all():
    print(f"Filtering {np.sum(~mask_pls)} pLS-T5 pairs with NaN/Inf")
    pls_feats, t5_feats, y_pls, disp_flag, w_pls = (
        pls_feats[mask_pls], t5_feats[mask_pls], y_pls[mask_pls],
        disp_flag[mask_pls], w_pls[mask_pls]
    )

X_pls_train, X_pls_test, X_t5raw_train, X_t5raw_test, y_pls_train, y_pls_test, w_pls_train, w_pls_test = train_test_split(
    pls_feats, t5_feats, y_pls, w_pls, test_size=0.20, random_state=42, stratify=y_pls, shuffle=True
)

pct_disp_pls = disp_flag.mean() * 100.0
print(f"pLS-T5 pairs -> train {len(y_pls_train)}  test {len(y_pls_test)}")
print(f"{pct_disp_pls:.2f}% of pLS-T5 pairs involve a displaced T5")


>>> Building pLS-T5 pairs (DR2 < 0.02) over 500 events ...
[evt    0]  pLSs= 6936  T5s= 7083  sim=1000  dis=1000
[evt    1]  pLSs= 6950  T5s= 6400  sim=1000  dis=1000
[evt    2]  pLSs= 7200  T5s= 6708  sim=1000  dis=1000
[evt    3]  pLSs= 7402  T5s= 7768  sim=1000  dis=1000
[evt    4]  pLSs= 6129  T5s= 5870  sim=1000  dis=1000
[evt    5]  pLSs= 7060  T5s= 7422  sim=1000  dis=1000
[evt    6]  pLSs= 6758  T5s= 6374  sim=1000  dis=1000
[evt    7]  pLSs= 8203  T5s= 9034  sim=1000  dis=1000
[evt    8]  pLSs= 7525  T5s= 8741  sim=1000  dis=1000
[evt    9]  pLSs= 8268  T5s= 7114  sim=1000  dis=1000
[evt   10]  pLSs= 6677  T5s= 6965  sim=1000  dis=1000
[evt   11]  pLSs= 6545  T5s= 6277  sim=1000  dis=1000
[evt   12]  pLSs= 7499  T5s= 7587  sim=1000  dis=1000
[evt   13]  pLSs= 8338  T5s= 8697  sim=1000  dis=1000
[evt   14]  pLSs= 6602  T5s= 6359  sim=1000  dis=1000
[evt   15]  pLSs= 7727  T5s= 8403  sim=1000  dis=1000
[evt   16]  pLSs= 6422  T5s= 5981  sim=1000  dis=1000
[evt   17]  pLSs= 5835

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class SiameseDataset(Dataset):
    def __init__(self, X_left, X_right, y, w):
        self.X_left  = torch.from_numpy(np.array(X_left).astype(np.float32))
        self.X_right = torch.from_numpy(np.array(X_right).astype(np.float32))
        self.y       = torch.from_numpy(np.array(y).astype(np.float32)).view(-1, 1)
        self.w       = torch.from_numpy(np.array(w).astype(np.float32)).view(-1, 1)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X_left[idx], self.X_right[idx], self.y[idx], self.w[idx]

class PLST5Dataset(Dataset):
    def __init__(self, pls, t5, y, w):
        self.pls = torch.from_numpy(pls)
        self.t5  = torch.from_numpy(t5)
        self.y   = torch.from_numpy(y.reshape(-1, 1).astype(np.float32))
        self.w   = torch.from_numpy(w.reshape(-1, 1).astype(np.float32))
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.pls[i], self.t5[i], self.y[i], self.w[i]

batch_size  = 1024
num_workers = min(os.cpu_count() or 4, 8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_t5_ds  = SiameseDataset(X_left_train,  X_right_train,  y_t5_train,  w_t5_train)
test_t5_ds   = SiameseDataset(X_left_test,   X_right_test,   y_t5_test,   w_t5_test)
train_pls_ds = PLST5Dataset(X_pls_train, X_t5raw_train, y_pls_train, w_pls_train)
test_pls_ds  = PLST5Dataset(X_pls_test,  X_t5raw_test,  y_pls_test,  w_pls_test)

on_gpu = device.type == 'cuda'

train_t5_loader  = DataLoader(train_t5_ds,  batch_size, shuffle=True,
                               num_workers=4 if on_gpu else 0,
                               pin_memory=on_gpu)
test_t5_loader   = DataLoader(test_t5_ds,   batch_size, shuffle=False,
                               num_workers=4 if on_gpu else 0,
                               pin_memory=on_gpu)
train_pls_loader = DataLoader(train_pls_ds, batch_size, shuffle=True,
                               num_workers=4 if on_gpu else 0,
                               pin_memory=on_gpu)
test_pls_loader  = DataLoader(test_pls_ds,  batch_size, shuffle=False,
                               num_workers=4 if on_gpu else 0,
                               pin_memory=on_gpu)

print(f"Loaders ready: T5 train {len(train_t5_ds)}, pLS-T5 train {len(train_pls_ds)}")

Loaders ready: T5 train 800000, pLS-T5 train 800000


In [12]:
import torch.nn as nn
import torch.optim as optim

# ── Improved architecture: wider (128→64→emb_dim), BatchNorm, no ReLU on final layer ──
# EMB_DIM increased from 6 to 16 for better representation capacity
EMB_DIM     = 16
T5_IN_DIM   = 30
PLS_IN_DIM  = 10

def _make_encoder(input_dim, emb_dim, final_relu=False):
    """Shared encoder factory: 128 -> 64 -> emb_dim, with BatchNorm.
    final_relu=False for Mahalanobis (avoids non-negative cone distortion).
    """
    layers = [
        nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(),
        nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(),
        nn.Linear(64, emb_dim),
    ]
    if final_relu:
        layers.append(nn.ReLU())
    return nn.Sequential(*layers)


# ── Euclidean networks ─────────────────────────────────────────────────────────
class EmbeddingNetT5(nn.Module):
    def __init__(self, input_dim=T5_IN_DIM, emb_dim=EMB_DIM):
        super().__init__()
        self.enc = _make_encoder(input_dim, emb_dim, final_relu=False)
    def forward(self, x): return self.enc(x)

class EmbeddingNetpLS(nn.Module):
    def __init__(self, input_dim=PLS_IN_DIM, emb_dim=EMB_DIM):
        super().__init__()
        self.enc = _make_encoder(input_dim, emb_dim, final_relu=False)
    def forward(self, x): return self.enc(x)


# ── Cosine networks ────────────────────────────────────────────────────────────
class CosEmbeddingNetT5(nn.Module):
    def __init__(self, input_dim=T5_IN_DIM, emb_dim=EMB_DIM):
        super().__init__()
        self.enc = _make_encoder(input_dim, emb_dim, final_relu=False)
    def forward(self, x): return self.enc(x)

class CosEmbeddingNetpLS(nn.Module):
    def __init__(self, input_dim=PLS_IN_DIM, emb_dim=EMB_DIM):
        super().__init__()
        self.enc = _make_encoder(input_dim, emb_dim, final_relu=False)
    def forward(self, x): return self.enc(x)


# ── Mahalanobis networks (NO final ReLU — critical for covariance geometry) ───
class MEmbeddingNetT5(nn.Module):
    def __init__(self, input_dim=T5_IN_DIM, emb_dim=EMB_DIM):
        super().__init__()
        # final_relu=False: embeddings can be negative, preserving full covariance structure
        self.enc = _make_encoder(input_dim, emb_dim, final_relu=False)
    def forward(self, x): return self.enc(x)

class MEmbeddingNetpLS(nn.Module):
    def __init__(self, input_dim=PLS_IN_DIM, emb_dim=EMB_DIM):
        super().__init__()
        self.enc = _make_encoder(input_dim, emb_dim, final_relu=False)
    def forward(self, x): return self.enc(x)


# ── Loss functions ─────────────────────────────────────────────────────────────
# Margin increased from 1.0 to 2.0: gives dissimilar pairs more room to separate
MARGIN = 2.0

class ContrastiveLoss(nn.Module):
    def __init__(self, margin=MARGIN):
        super().__init__()
        self.margin = margin
    def forward(self, d, label, weight=None):
        l_sim = (1 - label) * d.pow(2)
        l_dis = label * (self.margin - d).clamp(min=0.0).pow(2)
        loss = l_sim + l_dis
        if weight is not None:
            loss = loss * weight
        return loss.mean()

class CosineContrastiveLoss(nn.Module):
    def __init__(self, margin=MARGIN):
        super().__init__()
        self.margin = margin
    def forward(self, d, label, weight=None):
        l_sim = (1 - label) * d.pow(2)
        l_dis = label * (self.margin - d).clamp(min=0.0).pow(2)
        loss = l_sim + l_dis
        if weight is not None:
            loss = loss * weight
        return loss.mean()

criterion    = ContrastiveLoss(margin=MARGIN)
coscriterion = CosineContrastiveLoss(margin=MARGIN)


class MahalanobisMetric(nn.Module):
    """
    Learnable Mahalanobis distance with adaptive covariance
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.register_buffer('V', torch.eye(emb_dim))
        self.emb_dim = emb_dim
        
    def forward(self, x, y):
        diff = x - y
        # Mahalanobis distance: sqrt(diff^T * V * diff)
        dist_sq = torch.bmm(
            diff.unsqueeze(1),
            self.V.expand(diff.size(0), -1, -1) @ diff.unsqueeze(2)
        ).squeeze()
        return torch.sqrt(dist_sq.view(-1, 1) + 1e-6)
    
    def update_v(self, embeddings):
        if embeddings.shape[0] < self.emb_dim + 2:
            return
        cov = torch.cov(embeddings.t())
        psd_cov = cov + torch.eye(cov.size(0), device=cov.device) * 1e-3
        self.V = torch.linalg.inv(psd_cov)

print(f"Using device: {device}")

# Instantiate all six networks
embed_t5   = EmbeddingNetT5().to(device)
embed_pls  = EmbeddingNetpLS().to(device)
cembed_t5  = CosEmbeddingNetT5().to(device)
cembed_pls = CosEmbeddingNetpLS().to(device)
membed_t5  = MEmbeddingNetT5().to(device)
membed_pls = MEmbeddingNetpLS().to(device)

print(f"EmbeddingNetT5  params: {sum(p.numel() for p in embed_t5.parameters()):,}")
print(f"MEmbeddingNetT5 params: {sum(p.numel() for p in membed_t5.parameters()):,}")

Using device: cpu
EmbeddingNetT5  params: 13,648
MEmbeddingNetT5 params: 13,648


In [14]:
# ---------- Mahalanobis-only training ----------

# assumes models, m_metric, criterion, loader, optimizer, scheduler etc already defined
# and device set (cuda if available)

num_epochs = 20
COV_UPDATE_INTERVAL = 10

# Freeze Euclidean/Cosine models if defined
for p in embed_t5.parameters(): p.requires_grad = False
for p in embed_pls.parameters(): p.requires_grad = False
for p in cembed_t5.parameters(): p.requires_grad = False
for p in cembed_pls.parameters(): p.requires_grad = False

membed_t5.train(); membed_pls.train()

m_metric_t5 = MahalanobisMetric(EMB_DIM).to(device)
m_metric_pls = MahalanobisMetric(EMB_DIM).to(device)

mah_optimizer = optim.Adam(
    list(membed_t5.parameters()) + list(membed_pls.parameters()),
    lr=1e-3, weight_decay=1e-4
)
mah_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    mah_optimizer, mode='min', factor=0.5, patience=15
)

for epoch in range(1, num_epochs + 1):
    total_loss = 0.0
    sim_embs_buf = []

    for (l, r, y0, w0), (p5, t5f, y1, w1) in zip(train_t5_loader, train_pls_loader):
        l, r, y0_, w0_ = l.to(device), r.to(device), y0.to(device), w0.to(device)
        p5, t5f_, y1_, w1_ = p5.to(device), t5f.to(device), y1.to(device), w1.to(device)

        e_l_m, e_r_m = membed_t5(l), membed_t5(r)
        e_p_m, e_t_m = membed_pls(p5), membed_t5(t5f_)

        d0m = m_metric_t5(e_l_m, e_r_m)
        d1m = m_metric_pls(e_p_m, e_t_m)
        loss = criterion(d0m, y0_, w0_) + criterion(d1m, y1_, w1_)

        mah_optimizer.zero_grad()
        loss.backward()
        mah_optimizer.step()
        total_loss += loss.item()

        if epoch % COV_UPDATE_INTERVAL == 0:
            sim_t5 = (y0_.squeeze() == 0)
            if sim_t5.any():
                sim_embs_t5.append(e_l_m[sim_t5].detach())
                sim_embs_t5.append(e_r_m[sim_t5].detach())
            sim_pls = (y1_.squeeze() == 0)
            if sim_pls.any():
                sim_embs_pls.append(e_p_m[sim_pls].detach())
                sim_embs_pls.append(e_t_m[sim_pls].detach())

    if epoch % COV_UPDATE_INTERVAL == 0:
        if sim_embs_t5:
            all_sim_t5 = torch.cat(sim_embs_t5, dim=0)
            m_metric_t5.update_v(all_sim_t5)
        if sim_embs_pls:
            all_sim_pls = torch.cat(sim_embs_pls, dim=0)
            m_metric_pls.update_v(all_sim_pls)

    mah_scheduler.step(total_loss / len(train_pls_loader))

    if epoch % 5 == 0 or epoch == 1:
        print(f"[Mah] Epoch {epoch:3d}/{num_epochs}, loss = {total_loss/len(train_pls_loader):.4f}")

        cov_t5 = m_metric_t5.cov.detach().cpu() if hasattr(m_metric_t5, "cov") else None
        prec_t5 = m_metric_t5.V.detach().cpu() if hasattr(m_metric_t5, "V") else None
        cov_pls = m_metric_pls.cov.detach().cpu() if hasattr(m_metric_pls, "cov") else None
        prec_pls = m_metric_pls.V.detach().cpu() if hasattr(m_metric_pls, "V") else None

        print("  T5 covariance:", cov_t5)
        print("  T5 precision :", prec_t5)
        print("  pLS covariance:", cov_pls)
        print("  pLS precision :", prec_pls)

print("Mahalanobis-only training done.")

KeyboardInterrupt: 

In [ ]:
import torch.nn.functional as F

num_epochs = 50
CLIP_NORM  = 5.0
COV_UPDATE_INTERVAL = 10

eucl_optimizer = optim.Adam(
    list(embed_t5.parameters()) + list(embed_pls.parameters()),
    lr=1e-3, weight_decay=1e-4
)
eucl_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    eucl_optimizer, mode='min', factor=0.5, patience=15
)

cos_optimizer = optim.Adam(
    list(cembed_t5.parameters()) + list(cembed_pls.parameters()),
    lr=1e-3, weight_decay=1e-4
)
cos_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    cos_optimizer, mode='min', factor=0.5, patience=15
)

mah_optimizer = optim.Adam(
    list(membed_t5.parameters()) + list(membed_pls.parameters()),
    lr=1e-3, weight_decay=1e-4
)
mah_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    mah_optimizer, mode='min', factor=0.5, patience=15
)

print("Optimizers ready.")
print("=" * 60)
print("Training ALL metrics combined")
print("=" * 60)

for epoch in range(1, num_epochs + 1):
    embed_t5.train();  embed_pls.train()
    cembed_t5.train(); cembed_pls.train()
    membed_t5.train(); membed_pls.train()

    loss_e_total = loss_c_total = loss_m_total = 0.0
    sim_embs_buf = []

    for (l, r, y0, w0), (p5, t5f, y1, w1) in zip(train_t5_loader, train_pls_loader):
        l, r, y0_, w0_     = l.to(device), r.to(device), y0.to(device), w0.to(device)
        p5, t5f_, y1_, w1_ = p5.to(device), t5f.to(device), y1.to(device), w1.to(device)

        # --- Euclidean ---
        d0 = torch.sqrt(((embed_t5(l) - embed_t5(r))**2).sum(1, keepdim=True) + 1e-8)
        d1 = torch.sqrt(((embed_pls(p5) - embed_t5(t5f_))**2).sum(1, keepdim=True) + 1e-8)
        loss_e = criterion(d0, y0_, w0_) + criterion(d1, y1_, w1_)
        eucl_optimizer.zero_grad(); loss_e.backward(); eucl_optimizer.step()

        # --- Cosine ---
        d0c = 1.0 - F.cosine_similarity(cembed_t5(l), cembed_t5(r), dim=1).view(-1, 1)
        d1c = 1.0 - F.cosine_similarity(cembed_pls(p5), cembed_t5(t5f_), dim=1).view(-1, 1)
        loss_c = coscriterion(d0c, y0_, w0_) + coscriterion(d1c, y1_, w1_)
        cos_optimizer.zero_grad(); loss_c.backward(); cos_optimizer.step()

        # --- Mahalanobis ---
        e_l_m, e_r_m = membed_t5(l), membed_t5(r)
        e_p_m, e_t_m = membed_pls(p5), membed_t5(t5f_)
        d0m = m_metric(e_l_m, e_r_m)
        d1m = m_metric(e_p_m, e_t_m)
        loss_m = criterion(d0m, y0_, w0_) + criterion(d1m, y1_, w1_)
        mah_optimizer.zero_grad(); loss_m.backward(); mah_optimizer.step()

        loss_e_total += loss_e.item()
        loss_c_total += loss_c.item()
        loss_m_total += loss_m.item()

        # Collect similar embeddings for Mahalanobis V update
        if epoch % COV_UPDATE_INTERVAL == 0:
            sim_t5 = (y0_.squeeze() == 0)
            if sim_t5.any():
                sim_embs_buf.append(e_l_m[sim_t5].detach())
                sim_embs_buf.append(e_r_m[sim_t5].detach())
            sim_pls = (y1_.squeeze() == 0)
            if sim_pls.any():
                sim_embs_buf.append(e_p_m[sim_pls].detach())
                sim_embs_buf.append(e_t_m[sim_pls].detach())

    # Update Mahalanobis precision matrix
    if epoch % COV_UPDATE_INTERVAL == 0 and sim_embs_buf:
        all_sim = torch.cat(sim_embs_buf, dim=0)
        m_metric.update_v(all_sim)
        print(f"  [Epoch {epoch}] V updated from {all_sim.shape[0]} similar embeddings")

    n_batches = len(train_pls_loader)
    eucl_scheduler.step(loss_e_total / n_batches)
    cos_scheduler.step(loss_c_total / n_batches)
    mah_scheduler.step(loss_m_total / n_batches)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{num_epochs} | "
              f"Eucl: {loss_e_total/n_batches:.4f}  "
              f"Cos: {loss_c_total/n_batches:.4f}  "
              f"Mah: {loss_m_total/n_batches:.4f}")

print("Training complete.")

Optimizers ready.
Training ALL metrics combined


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import roc_curve, auc as sklearn_auc
import matplotlib.pyplot as plt

eucl_d_t5, mahal_d_t5, cos_d_t5, y_t5_list = [], [], [], []

embed_t5.eval();  embed_pls.eval()
cembed_t5.eval(); cembed_pls.eval()
membed_t5.eval(); membed_pls.eval()

with torch.no_grad():
    for (l, r, y, _w) in test_t5_loader:
        l, r = l.to(device), r.to(device)

        # Euclidean — uses embed_t5
        e_l_e = embed_t5(l);  e_r_e = embed_t5(r)
        d_eucl = torch.sqrt(((e_l_e - e_r_e)**2).sum(1, keepdim=True) + 1e-8)
        eucl_d_t5.append(d_eucl.cpu().numpy().ravel())

        # Mahalanobis — uses membed_t5 + m_metric  (separate network!)
        e_l_m = membed_t5(l); e_r_m = membed_t5(r)
        d_mah = m_metric(e_l_m, e_r_m)
        mahal_d_t5.append(d_mah.cpu().numpy().ravel())

        # Cosine — uses cembed_t5
        e_l_c = cembed_t5(l); e_r_c = cembed_t5(r)
        d_cos = 1.0 - torch.nn.functional.cosine_similarity(e_l_c, e_r_c, dim=1, eps=1e-8).view(-1, 1)
        cos_d_t5.append(d_cos.cpu().numpy().ravel())

        y_t5_list.append(y.numpy().ravel())

eucl_d_t5_arr  = np.concatenate(eucl_d_t5)
mahal_d_t5_arr = np.concatenate(mahal_d_t5)
cos_d_t5_arr   = np.concatenate(cos_d_t5)
y_t5_arr       = np.concatenate(y_t5_list)

eucl_d_pls, mahal_d_pls, cos_d_pls, y_pls_list = [], [], [], []

with torch.no_grad():
    for (p5, t5f, y, _w) in test_pls_loader:
        p5, t5f = p5.to(device), t5f.to(device)

        # Euclidean
        e_p_e = embed_pls(p5);  e_t_e = embed_t5(t5f)
        d_eucl = torch.sqrt(((e_p_e - e_t_e)**2).sum(1, keepdim=True) + 1e-8)
        eucl_d_pls.append(d_eucl.cpu().numpy().ravel())

        # Mahalanobis — uses membed_pls / membed_t5
        e_p_m = membed_pls(p5); e_t_m = membed_t5(t5f)
        d_mah = m_metric(e_p_m, e_t_m)
        mahal_d_pls.append(d_mah.cpu().numpy().ravel())

        # Cosine
        e_p_c = cembed_pls(p5); e_t_c = cembed_t5(t5f)
        d_cos = 1.0 - torch.nn.functional.cosine_similarity(e_p_c, e_t_c, dim=1, eps=1e-8).view(-1, 1)
        cos_d_pls.append(d_cos.cpu().numpy().ravel())

        y_pls_list.append(y.numpy().ravel())

eucl_d_pls_arr  = np.concatenate(eucl_d_pls)
mahal_d_pls_arr = np.concatenate(mahal_d_pls)
cos_d_pls_arr   = np.concatenate(cos_d_pls)
y_pls_arr       = np.concatenate(y_pls_list)

# Helper: if AUC < 0.5 the distance ordering is inverted; flip it
def safe_auc(fpr, tpr):
    a = sklearn_auc(fpr, tpr)
    return max(a, 1 - a)

fpr_e_t5,  tpr_e_t5,  _ = roc_curve(y_t5_arr,  eucl_d_t5_arr)
fpr_m_t5,  tpr_m_t5,  _ = roc_curve(y_t5_arr,  mahal_d_t5_arr)
fpr_c_t5,  tpr_c_t5,  _ = roc_curve(y_t5_arr,  cos_d_t5_arr)
fpr_e_pls, tpr_e_pls, _ = roc_curve(y_pls_arr, eucl_d_pls_arr)
fpr_m_pls, tpr_m_pls, _ = roc_curve(y_pls_arr, mahal_d_pls_arr)
fpr_c_pls, tpr_c_pls, _ = roc_curve(y_pls_arr, cos_d_pls_arr)

auc_e_t5  = safe_auc(fpr_e_t5,  tpr_e_t5)
auc_m_t5  = safe_auc(fpr_m_t5,  tpr_m_t5)
auc_c_t5  = safe_auc(fpr_c_t5,  tpr_c_t5)
auc_e_pls = safe_auc(fpr_e_pls, tpr_e_pls)
auc_m_pls = safe_auc(fpr_m_pls, tpr_m_pls)
auc_c_pls = safe_auc(fpr_c_pls, tpr_c_pls)

print(f"T5-T5   Euclidean   AUC: {auc_e_t5:.4f}")
print(f"T5-T5   Mahalanobis AUC: {auc_m_t5:.4f}")
print(f"T5-T5   Cosine      AUC: {auc_c_t5:.4f}")
print(f"\npLS-T5  Euclidean   AUC: {auc_e_pls:.4f}")
print(f"pLS-T5  Mahalanobis AUC: {auc_m_pls:.4f}")
print(f"pLS-T5  Cosine      AUC: {auc_c_pls:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot(fpr_e_t5, tpr_e_t5, label=f"Euclidean   (AUC={auc_e_t5:.3f})", linewidth=2)
axes[0].plot(fpr_m_t5, tpr_m_t5, label=f"Mahalanobis (AUC={auc_m_t5:.3f})", linewidth=2, linestyle='--')
axes[0].plot(fpr_c_t5, tpr_c_t5, label=f"Cosine      (AUC={auc_c_t5:.3f})", linewidth=2, linestyle=':')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("T5-T5 ROC Curve"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(fpr_e_pls, tpr_e_pls, label=f"Euclidean   (AUC={auc_e_pls:.3f})", linewidth=2)
axes[1].plot(fpr_m_pls, tpr_m_pls, label=f"Mahalanobis (AUC={auc_m_pls:.3f})", linewidth=2, linestyle='--')
axes[1].plot(fpr_c_pls, tpr_c_pls, label=f"Cosine      (AUC={auc_c_pls:.3f})", linewidth=2, linestyle=':')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("pLS-T5 ROC Curve"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("ROC curves saved to roc_curves.png")

In [ ]:
!git add -A && git commit -m "auto save" && git push

[main d1b79bd] auto save
 1 file changed, 39 insertions(+), 34 deletions(-)

(gnome-ssh-askpass:1880380): Gtk-WARNING **: 19:08:01.122: cannot open display: 
error: unable to read askpass response from '/usr/libexec/openssh/gnome-ssh-askpass'
Username for 'https://github.com': 